In [1]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# StreetSmart Safety Scoring – Model Training Notebook\n",
    "\n",
    "This notebook trains and evaluates the RandomForest safety scoring model.\n",
    "It generates synthetic data, trains the model, evaluates performance,\n",
    "and saves the final model to `../models/safety_model.pkl`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import pickle\n",
    "import os\n",
    "from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor\n",
    "from sklearn.model_selection import train_test_split, cross_val_score\n",
    "from sklearn.metrics import mean_absolute_error, r2_score\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "\n",
    "print('All imports successful')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ─── Generate Synthetic Training Data ───────────────────────────────────\n",
    "\n",
    "np.random.seed(42)\n",
    "N = 5000\n",
    "\n",
    "FEATURE_NAMES = [\n",
    "    'lat_mean', 'lng_mean', 'lat_std', 'lng_std',\n",
    "    'segment_count', 'hour_of_day', 'is_weekend',\n",
    "    'crime_density', 'light_level', 'crowd_density'\n",
    "]\n",
    "\n",
    "# Simulate realistic feature distributions\n",
    "lat_mean   = np.random.uniform(40.60, 40.85, N)    # NYC latitude range\n",
    "lng_mean   = np.random.uniform(-74.10, -73.90, N)  # NYC longitude range\n",
    "lat_std    = np.abs(np.random.normal(0.002, 0.001, N))\n",
    "lng_std    = np.abs(np.random.normal(0.002, 0.001, N))\n",
    "seg_count  = np.random.randint(5, 30, N).astype(float)\n",
    "hour       = np.random.randint(0, 24, N).astype(float)\n",
    "is_weekend = np.random.binomial(1, 0.29, N).astype(float)\n",
    "crime_den  = np.random.beta(2, 5, N)               # skewed toward low crime\n",
    "light_lvl  = np.random.beta(3, 2, N)               # skewed toward good lighting\n",
    "crowd_den  = np.random.beta(2, 3, N)\n",
    "\n",
    "X = np.column_stack([\n",
    "    lat_mean, lng_mean, lat_std, lng_std,\n",
    "    seg_count, hour, is_weekend,\n",
    "    crime_den, light_lvl, crowd_den\n",
    "])\n",
    "\n",
    "# Compute synthetic safety score\n",
    "# Higher safety = lower crime, better lighting, less crowd at night\n",
    "hour_penalty = np.where(hour >= 20, (hour - 20) * 3, 0)\n",
    "hour_penalty += np.where(hour < 6,  (6 - hour) * 2, 0)\n",
    "\n",
    "y = (\n",
    "    85\n",
    "    - crime_den * 40          # crime is bad\n",
    "    + light_lvl * 15          # lighting is good\n",
    "    - crowd_den * 8           # crowd is slightly bad\n",
    "    - hour_penalty            # nighttime penalty\n",
    "    + np.random.normal(0, 4, N)\n",
    ")\n",
    "y = np.clip(y, 15, 99)\n",
    "\n",
    "df = pd.DataFrame(X, columns=FEATURE_NAMES)\n",
    "df['safety_score'] = y\n",
    "\n",
    "print(f'Dataset shape: {df.shape}')\n",
    "print(df.describe().round(2))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ─── Visualize Target Distribution ──────────────────────────────────────\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "fig.patch.set_facecolor('#05080F')\n",
    "\n",
    "for ax in axes:\n",
    "    ax.set_facecolor('#0B1020')\n",
    "    ax.tick_params(colors='#8892B0')\n",
    "    for spine in ax.spines.values():\n",
    "        spine.set_edgecolor('#00E5FF')\n",
    "        spine.set_alpha(0.3)\n",
    "\n",
    "axes[0].hist(y, bins=50, color='#00FF9C', edgecolor='#05080F', alpha=0.85)\n",
    "axes[0].set_title('Safety Score Distribution', color='#E6F1FF')\n",
    "axes[0].set_xlabel('Safety Score', color='#8892B0')\n",
    "axes[0].set_ylabel('Count', color='#8892B0')\n",
    "\n",
    "axes[1].scatter(hour, y, alpha=0.08, s=5, color='#00E5FF')\n",
    "axes[1].set_title('Safety Score vs Hour of Day', color='#E6F1FF')\n",
    "axes[1].set_xlabel('Hour', color='#8892B0')\n",
    "axes[1].set_ylabel('Safety Score', color='#8892B0')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../notebooks/safety_distribution.png', dpi=150, bbox_inches='tight',\n",
    "            facecolor='#05080F')\n",
    "plt.show()\n",
    "print('Distribution plots saved')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ─── Train / Evaluate Model ──────────────────────────────────────────────\n",
    "\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "model = RandomForestRegressor(\n",
    "    n_estimators=150,\n",
    "    max_depth=12,\n",
    "    min_samples_split=4,\n",
    "    min_samples_leaf=2,\n",
    "    max_features='sqrt',\n",
    "    random_state=42,\n",
    "    n_jobs=-1\n",
    ")\n",
    "\n",
    "model.fit(X_train, y_train)\n",
    "\n",
    "y_pred = model.predict(X_test)\n",
    "mae = mean_absolute_error(y_test, y_pred)\n",
    "r2  = r2_score(y_test, y_pred)\n",
    "cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')\n",
    "\n",
    "print(f'Test MAE:    {mae:.3f}')\n",
    "print(f'Test R²:     {r2:.4f}')\n",
    "print(f'CV R² scores: {cv_scores.round(4)}')\n",
    "print(f'CV R² mean:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ─── Feature Importances ────────────────────────────────────────────────\n",
    "\n",
    "importances = model.feature_importances_\n",
    "indices = np.argsort(importances)[::-1]\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "fig.patch.set_facecolor('#05080F')\n",
    "ax.set_facecolor('#0B1020')\n",
    "ax.tick_params(colors='#8892B0')\n",
    "for spine in ax.spines.values():\n",
    "    spine.set_edgecolor('#00E5FF')\n",
    "    spine.set_alpha(0.3)\n",
    "\n",
    "colors = ['#00FF9C' if importances[i] > 0.15 else '#00E5FF' if importances[i] > 0.08 else '#8892B0'\n",
    "          for i in indices]\n",
    "\n",
    "ax.bar(range(len(FEATURE_NAMES)),\n",
    "       [importances[i] for i in indices],\n",
    "       color=colors, edgecolor='#05080F')\n",
    "ax.set_xticks(range(len(FEATURE_NAMES)))\n",
    "ax.set_xticklabels([FEATURE_NAMES[i] for i in indices],\n",
    "                   rotation=35, ha='right', color='#8892B0', fontsize=9)\n",
    "ax.set_title('Feature Importances – Safety Scoring Model', color='#E6F1FF', fontsize=13)\n",
    "ax.set_ylabel('Importance', color='#8892B0')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../notebooks/feature_importances.png', dpi=150, bbox_inches='tight',\n",
    "            facecolor='#05080F')\n",
    "plt.show()\n",
    "print('Feature importance chart saved')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ─── Save Model ──────────────────────────────────────────────────────────\n",
    "\n",
    "model_dir = '../models'\n",
    "os.makedirs(model_dir, exist_ok=True)\n",
    "model_path = os.path.join(model_dir, 'safety_model.pkl')\n",
    "\n",
    "with open(model_path, 'wb') as f:\n",
    "    pickle.dump(model, f)\n",
    "\n",
    "print(f'Model saved to: {model_path}')\n",
    "print(f'Model size: {os.path.getsize(model_path) / 1024:.1f} KB')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ─── Quick Inference Test ────────────────────────────────────────────────\n",
    "\n",
    "test_cases = [\n",
    "    {'hour': 14, 'crime': 0.1, 'light': 0.9, 'crowd': 0.5, 'label': 'Safe afternoon'},\n",
    "    {'hour': 23, 'crime': 0.7, 'light': 0.2, 'crowd': 0.2, 'label': 'Dangerous night'},\n",
    "    {'hour': 8,  'crime': 0.2, 'light': 0.8, 'crowd': 0.8, 'label': 'Rush hour morning'},\n",
    "    {'hour': 20, 'crime': 0.4, 'light': 0.5, 'crowd': 0.4, 'label': 'Evening commute'},\n",
    "]\n",
    "\n",
    "print('Inference test results:')\n",
    "print('-' * 55)\n",
    "for tc in test_cases:\n",
    "    features = np.array([[\n",
    "        40.75, -74.00, 0.002, 0.002, 15,\n",
    "        tc['hour'], 0,\n",
    "        tc['crime'], tc['light'], tc['crowd']\n",
    "    ]])\n",
    "    score = model.predict(features)[0]\n",
    "    risk = 'LOW' if score >= 75 else 'MEDIUM' if score >= 50 else 'HIGH'\n"

_IncompleteInputError: incomplete input (1896504663.py, line 254)